In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Day11").getOrCreate()

data = [
    (1, "login"),
    (2, "purchase")
]

df = spark.createDataFrame(data, ["user_id", "event_type"])

df.write.format("delta").saveAsTable("events")

In [0]:
new_data = [
    (3, "logout"),
    (4, "purchase")
]

new_df = spark.createDataFrame(new_data, ["user_id", "event_type"])

new_df.write.format("delta").mode("append").saveAsTable("events")

In [0]:
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("events") \
    .show()

+-------+----------+
|user_id|event_type|
+-------+----------+
|      1|     login|
|      2|  purchase|
+-------+----------+



In [0]:
# Time Travel
spark.read.format("delta") \
    .table("events") \
    .show()

+-------+----------+
|user_id|event_type|
+-------+----------+
|      3|    logout|
|      4|  purchase|
|      1|     login|
|      2|  purchase|
+-------+----------+



In [0]:
spark.sql("""
DELETE FROM events
WHERE user_id = 4
""")

DataFrame[num_affected_rows: bigint]

In [0]:
display(spark.sql("DESCRIBE HISTORY events"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-03-03T03:55:05.000Z,73807756678194,keerthi.amulya.1999@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2811636198371056),8a87c67c-0927-450c-9bc5-3d1baf944f2c,0303-034729-lyco4lz5-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 1800, p25FileSize -> 913, numDeletionVectorsRemoved -> 1, minFileSize -> 913, numAddedFiles -> 1, maxFileSize -> 913, p75FileSize -> 913, p50FileSize -> 913, numAddedBytes -> 913)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-03T03:55:02.000Z,73807756678194,keerthi.amulya.1999@gmail.com,DELETE,"Map(predicate -> [""(user_id#13691L = 4)""])",null,List(2811636198371056),8a87c67c-0927-450c-9bc5-3d1baf944f2c,0303-034729-lyco4lz5-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2685, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1921, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 685)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-03T03:49:17.000Z,73807756678194,keerthi.amulya.1999@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2811636198371056),f8ad30a7-c7b3-4876-8237-6d9c679df6f5,0303-034729-lyco4lz5-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 902)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-03T03:48:47.000Z,73807756678194,keerthi.amulya.1999@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2811636198371056),62892c8b-0251-40fa-8362-91d61b5e8035,0303-034729-lyco4lz5-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 898)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
spark.sql("""
INSERT INTO events
SELECT *
FROM events VERSION AS OF 1
WHERE user_id = 4
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]